##### Project 2 — Final Frozen Test Evaluation

##### Objective

Perform the final out-of-sample evaluation of the selected Triage Latency
prediction system on the frozen temporal test set.

##### Selected model

One-stage Ridge Regression with:

- log1p-transformed target
- narrative TF-IDF features
- company TF-IDF/one-hot representation
- intake-time features
- time features

##### Important methodological rule

The test set was frozen before final evaluation.

No:

- feature tuning
- hyperparameter tuning
- threshold tuning
- model selection
- architecture selection

will be performed using test results.

> The purpose of this notebook is measurement, not optimization.

##### Evaluation hierarchy

1. Overall regression performance
2. Typical-case performance
3. Long-delay/tail performance
4. Temporal robustness
5. Company robustness
6. Error direction and calibration
7. Comparison against validation expectations
8. Final go/no-go decision

In [3]:
## Imports
import numpy as np
import pandas as pd
from pathlib import Path

from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [4]:
PROJECT_ROOT = Path.cwd().parent.parent
# RAW_PATH = Path(r"D:\GURU_PROJECTS\CFPB-Financial-Consumer-Complaint-Intelligence\Data\complaints-cfpb-raw.csv")
DATA_PATH = PROJECT_ROOT / "Data" /"raw"/ "complaints-cfpb-raw.csv"

df = pd.read_csv(DATA_PATH)

df.shape

(81946, 16)

In [5]:
## Re-Construct Target
DATE_RECEIVED_COL = "Date received"
DATE_SENT_COL = "Date sent to company"
TEXT_COL = "Consumer complaint narrative"
COMPANY_COL = "Company"

df[DATE_RECEIVED_COL] = pd.to_datetime(
    df[DATE_RECEIVED_COL],
    errors="coerce",
    utc=True
)

df[DATE_SENT_COL] = pd.to_datetime(
    df[DATE_SENT_COL],
    errors="coerce",
    utc=True
)

df["triage_delay_days"] = (
    df[DATE_SENT_COL] - df[DATE_RECEIVED_COL]
).dt.total_seconds() / 86400

df = df.sort_values(DATE_RECEIVED_COL).reset_index(drop=True)

df[[
    DATE_RECEIVED_COL,
    DATE_SENT_COL,
    "triage_delay_days"
]].head()

,Date received,Date sent to company,triage_delay_days
0,2026-03-01 00:41:56+00:00,2026-03-19 11:33:43+00:00,18.4526
1,2026-03-01 01:04:12+00:00,2026-03-19 12:47:13+00:00,18.4882
2,2026-03-01 01:09:50+00:00,2026-03-19 11:29:47+00:00,18.4305
3,2026-03-01 01:37:46+00:00,2026-03-19 12:04:53+00:00,18.4355
4,2026-03-01 01:54:43+00:00,2026-03-19 11:36:29+00:00,18.4040


In [6]:
## Recreate Frozen Temporal Split
n = len(df)

train_end = int(n * 0.70)
val_end = train_end + int(n * 0.15)

train = df.iloc[:train_end].copy()
val = df.iloc[train_end:val_end].copy()
test = df.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

print("\nDate ranges")

for name, data in {
    "Train": train,
    "Validation": val,
    "Test": test
}.items():
    print(
        f"{name}: "
        f"{data[DATE_RECEIVED_COL].min()} → "
        f"{data[DATE_RECEIVED_COL].max()}"
    )

Train: (57362, 17)
Validation: (12291, 17)
Test: (12293, 17)

Date ranges
Train: 2026-03-01 00:41:56+00:00 → 2026-05-27 18:08:25+00:00
Validation: 2026-05-27 18:09:09+00:00 → 2026-06-17 14:32:33+00:00
Test: 2026-06-17 14:33:53+00:00 → 2026-08-11 14:27:33+00:00


In [7]:
## Verify Frozen Test Boundary
assert train[DATE_RECEIVED_COL].max() < val[DATE_RECEIVED_COL].min()
assert val[DATE_RECEIVED_COL].max() < test[DATE_RECEIVED_COL].min()

print("Temporal ordering verified.")

Temporal ordering verified.


In [8]:
## Freeze Selected Architecture
PRIMARY_THRESHOLD = 7

SELECTED_MODEL = {
    "architecture": "one-stage",
    "model": "Ridge",
    "target_transform": "log1p",
    "text_features": "TF-IDF",
    "company_features": "Company categorical representation",
}

SELECTED_MODEL

{'architecture': 'one-stage',
 'model': 'Ridge',
 'target_transform': 'log1p',
 'text_features': 'TF-IDF',
 'company_features': 'Company categorical representation'}

In [9]:
## Feature Engineering Function
def create_time_features(data):
    out = pd.DataFrame(index=data.index)

    dt = data[DATE_RECEIVED_COL]

    out["hour"] = dt.dt.hour
    out["day_of_week"] = dt.dt.dayofweek
    out["day_of_month"] = dt.dt.day
    out["month"] = dt.dt.month
    out["is_weekend"] = (dt.dt.dayofweek >= 5).astype(int)

    return out

##### Prepare Train + Validation

- Since the architecture is now frozen, we combine train and validation for final model fitting.

In [10]:
train_val = pd.concat(
    [train, val],
    axis=0
).reset_index(drop=True)

print(train_val.shape)

(69653, 17)


In [11]:
## Target 
y_train_val = train_val["triage_delay_days"].values
y_test = test["triage_delay_days"].values

y_train_val_log = np.log1p(y_train_val)

print("Train + Validation target:")
print(pd.Series(y_train_val).describe())

print("\nTest target:")
print(pd.Series(y_test).describe())

Train + Validation target:
count   69653.0000
mean        2.0205
std         8.6899
min         0.0002
25%         0.0050
50%         0.0099
75%         0.0194
max       146.7916
dtype: float64

Test target:
count   12293.0000
mean        0.1819
std         1.4754
min         0.0002
25%         0.0055
50%         0.0101
75%         0.0185
max        54.9815
dtype: float64


In [12]:
## Text Representation
train_val_text = train_val[TEXT_COL].fillna("")
test_text = test[TEXT_COL].fillna("")

text_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=100000,
    sublinear_tf=True
)

X_train_val_text = text_vectorizer.fit_transform(train_val_text)
X_test_text = text_vectorizer.transform(test_text)

print("Train+Val text:", X_train_val_text.shape)
print("Test text:", X_test_text.shape)

Train+Val text: (69653, 100000)
Test text: (12293, 100000)


In [13]:
## Company Representation
company_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2
)

X_train_val_company = company_vectorizer.fit_transform(
    train_val[COMPANY_COL].fillna("")
)

X_test_company = company_vectorizer.transform(
    test[COMPANY_COL].fillna("")
)

print("Company features:", X_train_val_company.shape)

Company features: (69653, 20830)


In [14]:
## Time Features
train_val_time = create_time_features(train_val)
test_time = create_time_features(test)

print(train_val_time.head())

   hour  day_of_week  day_of_month  month  is_weekend
0     0            6             1      3           1
1     1            6             1      3           1
2     1            6             1      3           1
3     1            6             1      3           1
4     1            6             1      3           1


In [15]:
## Convert Time Features into Sparse Features
from scipy.sparse import csr_matrix

X_train_val_time = csr_matrix(
    train_val_time.values
)

X_test_time = csr_matrix(
    test_time.values
)

In [16]:
## Final Feature Matrix
X_train_val = hstack([
    X_train_val_text,
    X_train_val_company,
    X_train_val_time
]).tocsr()

X_test = hstack([
    X_test_text,
    X_test_company,
    X_test_time
]).tocsr()

print("Final Train+Val matrix:", X_train_val.shape)
print("Final Test matrix:", X_test.shape)

Final Train+Val matrix: (69653, 120835)
Final Test matrix: (12293, 120835)


In [ ]:
## Train Final Model
final_model = Ridge(
    alpha=1.0 # if here alpha is hyperparameter then should we do hp tuning?
)

final_model.fit(
    X_train_val,
    y_train_val_log
)

print("Final model trained.")

Final model trained.


In [18]:
## Test Predictions
test_pred_log = final_model.predict(X_test)

test_pred = np.maximum(
    np.expm1(test_pred_log),
    0
)

print(pd.Series(test_pred).describe())

count   12293.0000
mean        0.3288
std         0.9484
min         0.0000
25%         0.0000
50%         0.1096
75%         0.3980
max        27.9002
dtype: float64


In [19]:
## Overall Test Metrics
test_metrics = pd.Series({
    "MAE": mean_absolute_error(y_test, test_pred),
    "MedianAE": median_absolute_error(y_test, test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
    "R2": r2_score(y_test, test_pred)
})

test_metrics


MAE         0.4512
MedianAE    0.1054
RMSE        1.6663
R2         -0.2757
dtype: float64

##### Baseline on Test

- We should compare against the same simple baseline, not merely report the model score

In [20]:
train_val_median = np.median(y_train_val)

baseline_pred = np.full(
    len(y_test),
    train_val_median
)

baseline_test_metrics = pd.Series({
    "MAE": mean_absolute_error(y_test, baseline_pred),
    "MedianAE": median_absolute_error(y_test, baseline_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, baseline_pred)),
    "R2": r2_score(y_test, baseline_pred)
})

baseline_test_metrics

MAE         0.1764
MedianAE    0.0056
RMSE        1.4853
R2         -0.0136
dtype: float64

In [21]:
## Model vs Baseline
test_comparison = pd.DataFrame({
    "Baseline": baseline_test_metrics,
    "Final One-stage Ridge": test_metrics
})

test_comparison

,Baseline,Final One-stage Ridge
MAE,0.1764,0.4512
MedianAE,0.0056,0.1054
RMSE,1.4853,1.6663
R2,-0.0136,-0.2757


In [22]:
## Improvement
improvement = pd.Series({
    "MAE improvement": (
        baseline_test_metrics["MAE"]
        - test_metrics["MAE"]
    ),
    
    "MedianAE improvement": (
        baseline_test_metrics["MedianAE"]
        - test_metrics["MedianAE"]
    ),
    
    "RMSE improvement": (
        baseline_test_metrics["RMSE"]
        - test_metrics["RMSE"]
    ),
    
    "R2 improvement": (
        test_metrics["R2"]
        - baseline_test_metrics["R2"]
    )
})

improvement

MAE improvement        -0.2748
MedianAE improvement   -0.0998
RMSE improvement       -0.1810
R2 improvement         -0.2621
dtype: float64

In [23]:
## Prediction Error
test_results = test[[
    DATE_RECEIVED_COL,
    COMPANY_COL,
    TEXT_COL,
    "triage_delay_days"
]].copy()

test_results["predicted_delay_days"] = test_pred

test_results["error"] = (
    test_results["triage_delay_days"]
    - test_results["predicted_delay_days"]
)

test_results["absolute_error"] = (
    test_results["error"].abs()
)

test_results.head()

,Date received,Company,Consumer complaint narrative,triage_delay_days,predicted_delay_days,error,absolute_error
69653,2026-06-17 14:33:53+00:00,"Westlake Services, LLC",My name is XXXX XXXX I got a XXXX XXXX from We...,0.0054,0.3889,-0.3835,0.3835
69654,2026-06-17 14:34:00+00:00,"CITIBANK, N.A.",I am the victim of fraud on my Citibank checki...,0.0104,0.0000,0.0104,0.0104
69655,2026-06-17 14:34:12+00:00,"SUNRISE CREDIT SERVICES, INC",SUNRISE CREDIT SERVICE is falsely reporting on...,0.0027,0.1078,-0.1051,0.1051
69656,2026-06-17 14:37:22+00:00,Unifin Inc.,I repeatedly received texts from a company cal...,0.0120,0.1576,-0.1456,0.1456
69657,2026-06-17 14:38:39+00:00,"Roosen, Varchetti & Olivier, PLLC",I am filing this complaint regarding an allege...,0.0059,0.6758,-0.6699,0.6699


In [24]:
## Directional Error
test_results["error_direction"] = np.select(
    [
        test_results["error"] > 0,
        test_results["error"] < 0
    ],
    [
        "Underpredicted",
        "Overpredicted"
    ],
    default="Exact"
)

direction_summary = (
    test_results["error_direction"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("percentage")
)

direction_summary

error_direction
Overpredicted    59.9121
Underpredicted   40.0879
Name: percentage, dtype: float64

In [25]:
## Delay Regimes
def delay_regime(x):
    if x <= 0.01:
        return "<=0.01d"
    elif x <= 0.05:
        return "0.01-0.05d"
    elif x <= 1:
        return "0.05-1d"
    elif x <= 7:
        return "1-7d"
    elif x <= 14:
        return "7-14d"
    elif x <= 30:
        return "14-30d"
    else:
        return ">30d"

test_results["delay_regime"] = (
    test_results["triage_delay_days"]
    .apply(delay_regime)
)

test_results["delay_regime"].value_counts()

delay_regime
<=0.01d       6101
0.01-0.05d    5507
0.05-1d        436
1-7d           136
7-14d          100
14-30d           8
>30d             5
Name: count, dtype: int64

In [26]:
## Error By Regime
regime_metrics = (
    test_results
    .groupby("delay_regime")
    .apply(
        lambda g: pd.Series({
            "n": len(g),
            "percentage": len(g) / len(test_results) * 100,
            "actual_mean": g["triage_delay_days"].mean(),
            "predicted_mean": g["predicted_delay_days"].mean(),
            "MAE": g["absolute_error"].mean(),
            "MedianAE": g["absolute_error"].median(),
            "RMSE": np.sqrt(
                np.mean(g["error"] ** 2)
            )
        }),
        include_groups=False
    )
)

regime_metrics

,n,percentage,actual_mean,predicted_mean,MAE,MedianAE,RMSE
delay_regime,,,,,,,
0.01-0.05d,5507.0000,44.7979,0.0198,0.3309,0.3247,0.1177,0.8427
0.05-1d,436.0000,3.5467,0.2087,0.4981,0.5109,0.1760,2.1649
1-7d,136.0000,1.1063,4.5088,1.7626,4.4900,4.4918,5.2585
14-30d,8.0000,0.0651,17.8180,0.1456,17.6724,17.4683,17.8133
7-14d,100.0000,0.8135,10.1981,1.3185,9.4456,9.6165,9.7233
<=0.01d,6101.0000,49.6299,0.0054,0.2664,0.2651,0.0722,0.5928
>30d,5.0000,0.0407,45.4094,0.9615,44.4479,49.0663,45.3481


In [27]:
## Tail Threshold Evaluation
thresholds = [1, 3, 7, 14, 30, 60]

tail_rows = []

for threshold in thresholds:

    mask = test_results["triage_delay_days"] > threshold

    if mask.sum() == 0:
        continue

    actual = test_results.loc[
        mask, "triage_delay_days"
    ]

    pred = test_results.loc[
        mask, "predicted_delay_days"
    ]

    tail_rows.append({
        "threshold_days": threshold,
        "n": mask.sum(),
        "percentage": mask.mean() * 100,
        "MAE": mean_absolute_error(actual, pred),
        "MedianAE": median_absolute_error(actual, pred),
        "RMSE": np.sqrt(mean_squared_error(actual, pred))
    })

tail_metrics = pd.DataFrame(tail_rows)

tail_metrics

,threshold_days,n,percentage,MAE,MedianAE,RMSE
0,1,249,2.0255,7.7061,6.3894,10.2255
1,3,217,1.7652,8.4228,6.8536,10.7972
2,7,113,0.9192,11.5768,10.1780,14.0401
3,14,13,0.1058,27.9707,19.8444,31.4040
4,30,5,0.0407,44.4479,49.0663,45.3481


In [28]:
## Important Tail Caveat
test_tail_counts = pd.Series({
    f">{t}d": (y_test > t).sum()
    for t in thresholds
})

test_tail_counts

>1d     249
>3d     217
>7d     113
>14d     13
>30d      5
>60d      0
dtype: int64

##### Interpretation rule

Very small test-set tail populations must not be treated as stable
performance estimates.

In particular:

- >14d has only a small number of observations
- >30d has very few observations
- >60d may have zero observations

- Therefore tail metrics are diagnostic evidence, not precise estimates
of production tail performance.

In [29]:
## > 7 Day Operational Analysis
actual_delayed = test_results["triage_delay_days"] > PRIMARY_THRESHOLD

predicted_delayed = (
    test_results["predicted_delay_days"] > PRIMARY_THRESHOLD
)

tp = (actual_delayed & predicted_delayed).sum()
fp = (~actual_delayed & predicted_delayed).sum()
fn = (actual_delayed & ~predicted_delayed).sum()
tn = (~actual_delayed & ~predicted_delayed).sum()

operational_matrix = pd.DataFrame(
    [
        [tn, fp],
        [fn, tp]
    ],
    index=["Actual <=7d", "Actual >7d"],
    columns=["Predicted <=7d", "Predicted >7d"]
)

operational_matrix

,Predicted <=7d,Predicted >7d
Actual <=7d,12146,34
Actual >7d,108,5


In [30]:
## Derived Detection Metrics
precision_7d = (
    tp / (tp + fp)
    if (tp + fp) > 0 else 0
)

recall_7d = (
    tp / (tp + fn)
    if (tp + fn) > 0 else 0
)

f1_7d = (
    2 * precision_7d * recall_7d /
    (precision_7d + recall_7d)
    if (precision_7d + recall_7d) > 0 else 0
)

test_7d_detection = pd.Series({
    "Precision": precision_7d,
    "Recall": recall_7d,
    "F1": f1_7d,
    "Flag rate": predicted_delayed.mean()
})

test_7d_detection

Precision   0.1282
Recall      0.0442
F1          0.0658
Flag rate   0.0032
dtype: float64

In [31]:
## Prediction Compression
prediction_distribution = pd.DataFrame({
    "Actual": pd.Series(y_test).describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ),
    
    "Predicted": pd.Series(test_pred).describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
})

prediction_distribution

,Actual,Predicted
count,12293.0000,12293.0000
mean,0.1819,0.3288
std,1.4754,0.9484
min,0.0002,0.0000
50%,0.0101,0.1096
90%,0.0332,0.7638
95%,0.0551,1.1107
99%,6.8140,2.6506
max,54.9815,27.9002


In [32]:
## Actual vs Predicted Quantiles
quantiles = [0.50, 0.75, 0.90, 0.95, 0.99, 1.00]

quantile_comparison = pd.DataFrame({
    "Actual": np.quantile(y_test, quantiles),
    "Predicted": np.quantile(test_pred, quantiles)
}, index=quantiles)

quantile_comparison.index.name = "quantile"

quantile_comparison

,Actual,Predicted
quantile,,
0.5000,0.0101,0.1096
0.7500,0.0185,0.3980
0.9000,0.0332,0.7638
0.9500,0.0551,1.1107
0.9900,6.8140,2.6506
1.0000,54.9815,27.9002


In [33]:
## Company Robustness
train_val_companies = set(
    train_val[COMPANY_COL]
    .dropna()
    .unique()
)

test_results["company_seen_in_training"] = (
    test_results[COMPANY_COL]
    .isin(train_val_companies)
)

test_results[
    "company_seen_in_training"
].value_counts()

company_seen_in_training
True     12195
False       98
Name: count, dtype: int64

In [34]:
## Seen vs Unseen Company Performance
company_robustness = (
    test_results
    .groupby("company_seen_in_training")
    .apply(
        lambda g: pd.Series({
            "n": len(g),
            "MAE": g["absolute_error"].mean(),
            "MedianAE": g["absolute_error"].median(),
            "RMSE": np.sqrt(
                np.mean(g["error"] ** 2)
            ),
            "mean_actual_delay": g[
                "triage_delay_days"
            ].mean()
        }),
        include_groups=False
    )
)

company_robustness

,n,MAE,MedianAE,RMSE,mean_actual_delay
company_seen_in_training,,,,,
False,98.0000,0.9487,0.6076,1.4309,0.0991
True,12195.0000,0.4472,0.1034,1.6681,0.1825


In [35]:
## Monthly Test Performance
test_results["month"] = (
    test_results[DATE_RECEIVED_COL]
    .dt.to_period("M")
    .astype(str)
)

monthly_test_metrics = (
    test_results
    .groupby("month")
    .apply(
        lambda g: pd.Series({
            "n": len(g),
            "actual_mean": g["triage_delay_days"].mean(),
            "predicted_mean": g["predicted_delay_days"].mean(),
            "MAE": g["absolute_error"].mean(),
            "MedianAE": g["absolute_error"].median(),
            "RMSE": np.sqrt(
                np.mean(g["error"] ** 2)
            )
        }),
        include_groups=False
    )
)

monthly_test_metrics

C:\Users\MY PC\AppData\Local\Temp\ipykernel_11912\2734022599.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


,n,actual_mean,predicted_mean,MAE,MedianAE,RMSE
month,,,,,,
2026-06,7131.0000,0.2780,0.3545,0.5507,0.1096,2.0593
2026-07,5161.0000,0.0490,0.2935,0.3139,0.1017,0.8686
2026-08,1.0000,0.0031,0.0000,0.0031,0.0031,0.0031


In [36]:
## Test Residual Summary
residual_summary = pd.Series({
    "mean_error": test_results["error"].mean(),
    "median_error": test_results["error"].median(),
    "mean_absolute_error": test_results["absolute_error"].mean(),
    "p90_absolute_error": test_results["absolute_error"].quantile(0.90),
    "p95_absolute_error": test_results["absolute_error"].quantile(0.95),
    "p99_absolute_error": test_results["absolute_error"].quantile(0.99),
    "max_absolute_error": test_results["absolute_error"].max()
})

residual_summary

mean_error            -0.1470
median_error          -0.0822
mean_absolute_error    0.4512
p90_absolute_error     0.8310
p95_absolute_error     1.3446
p99_absolute_error     7.3310
max_absolute_error    54.4723
dtype: float64

In [37]:
## Validation vs Final Test
validation_metrics = pd.Series({
    "MAE": 1.373852,
    "MedianAE": 0.212139,
    "RMSE": 5.364362,
    "R2": 0.011412
})

generalization_comparison = pd.DataFrame({
    "Validation": validation_metrics,
    "Frozen Test": test_metrics
})

generalization_comparison

,Validation,Frozen Test
MAE,1.3739,0.4512
MedianAE,0.2121,0.1054
RMSE,5.3644,1.6663
R2,0.0114,-0.2757


In [38]:
## Metric Drift
metric_change = pd.DataFrame({
    "Validation": validation_metrics,
    "Test": test_metrics
})

metric_change["Absolute Change"] = (
    metric_change["Test"]
    - metric_change["Validation"]
)

metric_change["Relative Change %"] = (
    metric_change["Absolute Change"]
    / metric_change["Validation"].replace(0, np.nan)
    * 100
)

metric_change

,Validation,Test,Absolute Change,Relative Change %
MAE,1.3739,0.4512,-0.9226,-67.1560
MedianAE,0.2121,0.1054,-0.1068,-50.3366
RMSE,5.3644,1.6663,-3.6980,-68.9371
R2,0.0114,-0.2757,-0.2871,-2515.7779


In [ ]:
## Save Test Predictions
OUTPUT_PATH = (
    "/mnt/data/project2_final_test_predictions.csv"
)

test_results.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved: {OUTPUT_PATH}")

In [ ]:
### Final Decision Record
final_decision = {
    "selected_architecture": "One-stage Ridge regression",
    "target": "triage_delay_days",
    "target_transform": "log1p",
    "primary_metric": "MAE",
    "secondary_metrics": [
        "MedianAE",
        "RMSE",
        "R2"
    ],
    "test_set_used_for_model_selection": False,
    "test_set_used_for_final_evaluation": True,
    "two_stage_selected": False,
    "test_evaluation_status": "COMPLETE"
}

pd.Series(final_decision)

#### Final Analysis Insights

#### 1. Generalization

Compare frozen validation performance against the final test result.

The key question is not:

> "Is the test score good?"

The better question is:

> "Did the selected model behave consistently enough on genuinely future data?"

---

#### 2. Typical-case performance

Evaluate MAE and MedianAE together.

MedianAE describes the typical complaint.

MAE captures the average operational error.

---

#### 3. Tail behavior

The model should not be expected to perfectly predict rare extreme delays.

However, systematic underprediction of long delays is operationally important.

---

#### 4. Operational interpretation

A useful model should help answer:

> "Which complaints are likely to experience unusually long processing delays?"

rather than simply producing a mathematically optimized average prediction.

---

#### 5. Model limitation

The one-stage model compresses predictions toward the center.

This means:

- ordinary cases are relatively easier
- long-delay cases are systematically underestimated
- rare operational bottlenecks remain difficult to predict

---

#### 6. Two-stage architecture

The two-stage architecture was investigated but not selected.

Although it substantially improved MedianAE, it worsened:

- overall MAE
- RMSE
- R²

Therefore it remains a candidate for future iteration, not the current production candidate.

---

#### 7. Final status

- The final model is now evaluated on the frozen temporal test set.

- No further model tuning should be performed using this test result.

##### Project 2 — Model Packaging

#### Objective

Convert the final selected experimental model into a reproducible,
versioned model artifact.

The packaged model represents the final one-stage Ridge architecture
evaluated during the frozen test phase.

#### Important status

Frozen test evaluation showed that the final Ridge model did not
outperform the simple train + validation median baseline.

Therefore this artifact is classified as:

EXPERIMENTAL / RESEARCH ARTIFACT

It is not approved for autonomous production deployment.

#### Package contents

1. Trained regression model
2. Text vectorizer
3. Company vectorizer
4. Feature configuration
5. Model metadata
6. Training-data metadata
7. Evaluation results
8. Prediction helper

In [39]:
### Imports
import json
import hashlib
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

In [41]:
## Create Package Directory
PROJECT_ROOT = Path.cwd().parent.parent
MODEL_DIR = PROJECT_ROOT / "Reports" / "project2"


MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DIR

WindowsPath('d:/GURU_PROJECTS/ML-FINTECH&BANK/Reports/project2')

In [42]:
### Model Configuration
MODEL_CONFIG = {
    "project": "CFPB Triage Latency Prediction",
    "version": "1.0.0",
    
    "architecture": "one-stage",
    "algorithm": "Ridge Regression",
    "target": "triage_delay_days",
    "target_transform": "log1p",
    
    "text_vectorizer": {
        "type": "TfidfVectorizer",
        "ngram_range": [1, 2],
        "min_df": 3,
        "max_features": 100000,
        "sublinear_tf": True
    },
    
    "company_vectorizer": {
        "type": "TfidfVectorizer",
        "analyzer": "char",
        "ngram_range": [2, 5],
        "min_df": 2
    },
    
    "regression": {
        "type": "Ridge",
        "alpha": 1.0
    },
    
    "primary_metric": "MAE",
    "secondary_metrics": [
        "MedianAE",
        "RMSE",
        "R2"
    ],
    
    "status": "experimental_not_production_approved"
}

MODEL_CONFIG

{'project': 'CFPB Triage Latency Prediction',
 'version': '1.0.0',
 'architecture': 'one-stage',
 'algorithm': 'Ridge Regression',
 'target': 'triage_delay_days',
 'target_transform': 'log1p',
 'text_vectorizer': {'type': 'TfidfVectorizer',
  'ngram_range': [1, 2],
  'min_df': 3,
  'max_features': 100000,
  'sublinear_tf': True},
 'company_vectorizer': {'type': 'TfidfVectorizer',
  'analyzer': 'char',
  'ngram_range': [2, 5],
  'min_df': 2},
 'regression': {'type': 'Ridge', 'alpha': 1.0},
 'primary_metric': 'MAE',
 'secondary_metrics': ['MedianAE', 'RMSE', 'R2'],
 'status': 'experimental_not_production_approved'}

In [43]:
## Save Configuration
config_path = MODEL_DIR / "model_config.json"

with open(config_path, "w") as f:
    json.dump(
        MODEL_CONFIG,
        f,
        indent=4
    )

print(config_path)

d:\GURU_PROJECTS\ML-FINTECH&BANK\Reports\project2\model_config.json


In [44]:
## Save Model
model_path = MODEL_DIR / "ridge_model.joblib"

joblib.dump(
    final_model,
    model_path
)

print(model_path)

d:\GURU_PROJECTS\ML-FINTECH&BANK\Reports\project2\ridge_model.joblib


In [45]:
## Save Vectorizers
text_vectorizer_path = (
    MODEL_DIR / "text_vectorizer.joblib"
)

company_vectorizer_path = (
    MODEL_DIR / "company_vectorizer.joblib"
)

joblib.dump(
    text_vectorizer,
    text_vectorizer_path
)

joblib.dump(
    company_vectorizer,
    company_vectorizer_path
)

print("Vectorizers saved.")

Vectorizers saved.


In [46]:
## Feature Metadata
feature_metadata = {
    "text_features": X_train_val_text.shape[1],
    "company_features": X_train_val_company.shape[1],
    "time_features": X_train_val_time.shape[1],
    "total_features": X_train_val.shape[1],
    
    "feature_order": [
        "narrative_tfidf",
        "company_tfidf",
        "time_features"
    ],
    
    "time_features": [
        "hour",
        "day_of_week",
        "day_of_month",
        "month",
        "is_weekend"
    ]
}

with open(
    MODEL_DIR / "feature_metadata.json",
    "w"
) as f:
    json.dump(
        feature_metadata,
        f,
        indent=4
    )

feature_metadata

{'text_features': 100000,
 'company_features': 20830,
 'time_features': ['hour',
  'day_of_week',
  'day_of_month',
  'month',
  'is_weekend'],
 'total_features': 120835,
 'feature_order': ['narrative_tfidf', 'company_tfidf', 'time_features']}

In [47]:
## Training Metadata
training_metadata = {
    "training_rows": int(len(train_val)),
    "training_date_min": str(
        train_val[DATE_RECEIVED_COL].min()
    ),
    "training_date_max": str(
        train_val[DATE_RECEIVED_COL].max()
    ),
    
    "test_rows": int(len(test)),
    "test_date_min": str(
        test[DATE_RECEIVED_COL].min()
    ),
    "test_date_max": str(
        test[DATE_RECEIVED_COL].max()
    ),
    
    "target_mean": float(
        train_val["triage_delay_days"].mean()
    ),
    "target_median": float(
        train_val["triage_delay_days"].median()
    ),
    
    "primary_threshold_days": 7
}

with open(
    MODEL_DIR / "training_metadata.json",
    "w"
) as f:
    json.dump(
        training_metadata,
        f,
        indent=4
    )

In [48]:
## Evaluation Metadata
evaluation_metadata = {
    "validation": {
        "MAE": 1.3739,
        "MedianAE": 0.2121,
        "RMSE": 5.3644,
        "R2": 0.0114
    },
    
    "frozen_test": {
        "MAE": 0.4512,
        "MedianAE": 0.1054,
        "RMSE": 1.6663,
        "R2": -0.2757
    },
    
    "test_baseline": {
        "MAE": 0.1764,
        "MedianAE": 0.0056,
        "RMSE": 1.4853,
        "R2": -0.0136
    },
    
    "deployment_status": (
        "NOT_PRODUCTION_APPROVED"
    ),
    
    "reason": (
        "Final Ridge model did not outperform "
        "the simple train+validation median baseline "
        "on frozen test data."
    )
}

with open(
    MODEL_DIR / "evaluation_metadata.json",
    "w"
) as f:
    json.dump(
        evaluation_metadata,
        f,
        indent=4
    )

In [49]:
## Prediction Function
def create_time_features(data):
    out = pd.DataFrame(index=data.index)

    dt = pd.to_datetime(
        data["Date received"],
        errors="coerce",
        utc=True
    )

    out["hour"] = dt.dt.hour
    out["day_of_week"] = dt.dt.dayofweek
    out["day_of_month"] = dt.dt.day
    out["month"] = dt.dt.month
    out["is_weekend"] = (
        dt.dt.dayofweek >= 5
    ).astype(int)

    return out


def predict_triage_delay(
    data,
    model,
    text_vectorizer,
    company_vectorizer
):

    text = data[
        "Consumer complaint narrative"
    ].fillna("")

    company = data[
        "Company"
    ].fillna("")

    X_text = text_vectorizer.transform(text)

    X_company = company_vectorizer.transform(
        company
    )

    time_features = create_time_features(data)

    X_time = csr_matrix(
        time_features.values
    )

    X = hstack([
        X_text,
        X_company,
        X_time
    ]).tocsr()

    log_prediction = model.predict(X)

    prediction = np.maximum(
        np.expm1(log_prediction),
        0
    )

    return prediction

In [50]:
## Import Sparse Matrix Helper
from scipy.sparse import csr_matrix

## Prediction Smoke Test
sample = test.head(10).copy()

sample_predictions = predict_triage_delay(
    sample,
    final_model,
    text_vectorizer,
    company_vectorizer
)

sample[
    [
        "Date received",
        "Company",
        "triage_delay_days"
    ]
].assign(
    predicted_delay_days=sample_predictions
)

,Date received,Company,triage_delay_days,predicted_delay_days
69653,2026-06-17 14:33:53+00:00,"Westlake Services, LLC",0.0054,0.3889
69654,2026-06-17 14:34:00+00:00,"CITIBANK, N.A.",0.0104,0.0000
69655,2026-06-17 14:34:12+00:00,"SUNRISE CREDIT SERVICES, INC",0.0027,0.1078
69656,2026-06-17 14:37:22+00:00,Unifin Inc.,0.0120,0.1576
69657,2026-06-17 14:38:39+00:00,"Roosen, Varchetti & Olivier, PLLC",0.0059,0.6758
69658,2026-06-17 14:38:42+00:00,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",0.0084,0.4745
69659,2026-06-17 14:40:46+00:00,PNC Bank N.A.,0.0238,0.1139
69660,2026-06-17 14:41:03+00:00,OneMain Finance Corporation,0.0363,0.6176
69661,2026-06-17 14:41:55+00:00,TRUIST FINANCIAL CORPORATION,0.0079,0.5623
69662,2026-06-17 14:42:43+00:00,Chime Financial Inc,0.0066,0.0000


In [51]:
## Validate Packaging Reproducibility

loaded_model = joblib.load(
    MODEL_DIR / "ridge_model.joblib"
)

loaded_text_vectorizer = joblib.load(
    MODEL_DIR / "text_vectorizer.joblib"
)

loaded_company_vectorizer = joblib.load(
    MODEL_DIR / "company_vectorizer.joblib"
)

In [52]:
## Reproduce Prediction
loaded_predictions = predict_triage_delay(
    test.head(100),
    loaded_model,
    loaded_text_vectorizer,
    loaded_company_vectorizer
)

original_predictions = predict_triage_delay(
    test.head(100),
    final_model,
    text_vectorizer,
    company_vectorizer
)

np.allclose(
    loaded_predictions,
    original_predictions
)

True

In [53]:
## Package Integrity
def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            sha.update(chunk)

    return sha.hexdigest()

In [54]:
## Generate Checksums
checksums = {}

for path in MODEL_DIR.iterdir():

    if path.is_file():
        checksums[path.name] = (
            sha256_file(path)
        )

checksums

{'company_vectorizer.joblib': '058ff678d46a7b0546d3972755cbf82cabc1b6152edd2965e615fe17230e286e',
 'evaluation_metadata.json': '3e4dba754f6daa5e82d820864a74c81ba1ee31fb1d035372d9039de9976623c9',
 'feature_metadata.json': '8b34fcd9039fe16c3fae55c60c34eb2d8b28b7cef6520253dba19586f55e86b5',
 'model_config.json': 'd6a7df5f19aefd5fc1b7b62814153ded5c1b8ba3db74b876b478d391c7503461',
 'project2_model_experiment_results.csv': 'd48c6eb116d7aec9c0ae89dac21867fa21bc0286c092d56aeec50af12e12bf26',
 'project2_tail_performance.csv': '483ccf0599fceb08d1a8dce1da59ff7ab537f5af6d20946dbeaec2a941f33435',
 'project2_target_audit_summary.json': '1bdd0b2e2f90c7a11f38748f51406894b739987909dca200f261b312672ce00c',
 'ridge_model.joblib': 'b02faaf63f8f536e741a03102e6445692569c981e2e0bb0a06d5e583dd359f95',
 'text_vectorizer.joblib': '630007e5b1d3b8497904944f7417af7afb8d7088d879757670c8e55b74b12024',
 'training_metadata.json': 'a822a68bc11365f39dbc762c34f998006c79fa1f1389ef1872b1018dffae4c5c'}

In [55]:
## Save CheckSums
with open(
    MODEL_DIR / "SHA256.json",
    "w"
) as f:
    json.dump(
        checksums,
        f,
        indent=4
    )

print("Checksums saved.")

Checksums saved.


In [56]:
## Package Manifest
manifest = pd.DataFrame({
    "file": [
        p.name
        for p in MODEL_DIR.iterdir()
        if p.is_file()
    ]
})

manifest

,file
0,company_vectorizer.joblib
1,evaluation_metadata.json
2,feature_metadata.json
3,model_config.json
4,project2_model_experiment_results.csv
5,project2_tail_performance.csv
6,project2_target_audit_summary.json
7,ridge_model.joblib
8,SHA256.json
9,text_vectorizer.joblib


In [57]:
## Final Packaging Gate
required_files = [
    "ridge_model.joblib",
    "text_vectorizer.joblib",
    "company_vectorizer.joblib",
    "model_config.json",
    "feature_metadata.json",
    "training_metadata.json",
    "evaluation_metadata.json",
    "SHA256.json"
]

missing_files = [
    f for f in required_files
    if not (MODEL_DIR / f).exists()
]

assert not missing_files, (
    f"Missing package files: {missing_files}"
)

print("MODEL PACKAGE INTEGRITY CHECK: PASSED")

MODEL PACKAGE INTEGRITY CHECK: PASSED


#### Bussiness Decision Layer
ML question:
"What delay will this complaint have?"

        ↓

Business question:
"What should operations do differently
because of this prediction?"

### Project 2 — Business Decision Layer

#### Objective

Translate model predictions into operational decisions while explicitly
accounting for:

- prediction uncertainty
- model error
- process drift
- workload
- false positives
- false negatives
- operational capacity
- human review

##### Current model status

The final one-stage Ridge model is NOT approved for autonomous
production decision-making.

Frozen test evaluation showed that the model underperformed the
simple train+validation median baseline.

Therefore the business layer is designed as a:

1. Decision framework
2. Human-in-the-loop prototype
3. Monitoring specification
4. Future production architecture

- rather than an autonomous deployment.

### Business Objective

The operational objective is not simply to minimize prediction error.

The organization wants to:

- identify complaints at risk of unusually long triage delay
- prioritize operational attention
- reduce avoidable processing bottlenecks
- protect service-level performance
- avoid overwhelming investigators with false alerts

Therefore:

Prediction → Risk signal → Operational action

- must be evaluated as a complete system.

### Decision architecture
business_flow = """
Complaint Received
        |
        v
Prediction Available
        |
        v
Delay Risk Assessment
        |
        +-----------------------+
        |                       |
     Low Risk               High Risk
        |                       |
   Normal Queue          Human Review Queue
                                |
                                v
                       Capacity / Root Cause
                                |
                                v
                         Operational Action
                                |
                                v
                         Outcome Monitoring
                                |
                                v
                         Model Feedback
"""
print(business_flow)

#### Important Distinction
#### Prediction ≠ Decision

A model prediction should not automatically trigger an operational
action.

For example:

Predicted delay = 8 days

does NOT necessarily mean:- "Escalate immediately."

The organization needs additional context:

- SLA threshold
- complaint age
- operational capacity
- customer/business priority
- current queue
- confidence
- model reliability
- reason for prediction

Therefore the correct architecture is:

Prediction
→ Risk
→ Policy
→ Human/Operational Decision
→ Action

In [58]:
## Operational Risk Bands
def operational_band(delay_prediction):

    if delay_prediction <= 1:
        return "NORMAL"

    elif delay_prediction <= 3:
        return "WATCH"

    elif delay_prediction <= 7:
        return "ELEVATED"

    else:
        return "HIGH_RISK"

In [59]:
## Apply Bands
business_view = test_results.copy()

business_view["predicted_risk_band"] = (
    business_view["predicted_delay_days"]
    .apply(operational_band)
)

business_view[
    [
        "triage_delay_days",
        "predicted_delay_days",
        "predicted_risk_band"
    ]
].head()

,triage_delay_days,predicted_delay_days,predicted_risk_band
69653,0.0054,0.3889,NORMAL
69654,0.0104,0.0000,NORMAL
69655,0.0027,0.1078,NORMAL
69656,0.0120,0.1576,NORMAL
69657,0.0059,0.6758,NORMAL


##### Governance Rule

The risk bands above are a business-policy prototype.

They must NOT be interpreted as validated operational thresholds
until the model demonstrates sufficient out-of-sample performance.

- This distinction prevents:

> "model predicted high risk"

- from becoming:

> "customer definitely requires escalation."

### Baseline Policy

Current evidence suggests that a simple historical baseline is more
reliable than the trained Ridge model for point prediction on the
frozen test period.

Therefore the baseline should remain the benchmark for future models.

> Any future model must demonstrate:

- Model benefit > Baseline benefit

- before production consideration.

In [60]:
### Model Promotion Gate
promotion_gate = {
    "must_beat_baseline_mae": True,
    "must_beat_baseline_medianAE": True,
    "must_not_materially_worsen_RMSE": True,
    "must_demonstrate_temporal_robustness": True,
    "must_validate_tail_behavior": True,
    "must_pass_company_robustness": True,
    "must_have_monitoring_plan": True,
    "must_have_human_override": True
}

pd.Series(promotion_gate)

must_beat_baseline_mae                  True
must_beat_baseline_medianAE             True
must_not_materially_worsen_RMSE         True
must_demonstrate_temporal_robustness    True
must_validate_tail_behavior             True
must_pass_company_robustness            True
must_have_monitoring_plan               True
must_have_human_override                True
dtype: bool

### Business KPIs

### Model KPIs

- MAE
- Median Absolute Error
- RMSE
- R²
- >7-day detection recall
- >7-day detection precision
- calibration / reliability

### Operational KPIs

- median triage time
- P90 triage time
- P95 triage time
- percentage >7 days
- percentage >14 days
- percentage >30 days
- queue backlog
- SLA breach rate

### Efficiency KPIs

- alerts generated
- alert rate
- alerts reviewed
- useful-alert rate
- false-alert rate
- analyst hours per useful alert

### Outcome KPIs

- delayed complaints resolved
- reduction in long-delay rate
- SLA compliance
- customer-impact indicators

### Decision Matrix 
| Model Signal | Operational Meaning | Suggested Action |
|---|---|---|
| Low predicted delay | Normal processing expected | Standard queue |
| Moderate predicted delay | Potential bottleneck | Monitor |
| High predicted delay | Possible SLA risk | Human review |
| High prediction + high queue load | Elevated operational risk | Prioritize capacity |
| High prediction but low confidence | Uncertain | Human verification |
| Model unavailable | No ML decision | Fall back to baseline |

### Human-in-the-Loop Architecture

The model should recommend.

The operational team decides.

Architecture:

Model
  ↓
Risk signal
  ↓
Reason / evidence
  ↓
Human review
  ↓
Approve / override
  ↓
Operational action
  ↓
Outcome logged

In [61]:
### FallBack Policy
def decision_with_fallback(
    model_available,
    prediction,
    baseline_prediction
):

    if not model_available:
        return {
            "prediction": baseline_prediction,
            "source": "baseline",
            "action": "STANDARD_POLICY"
        }

    return {
        "prediction": prediction,
        "source": "ml_model",
        "action": operational_band(prediction)
    }

### Production Monitoring Architecture

The system should monitor four different things.

### 1. Data Drift

Are incoming complaints changing?

Monitor:

- Product distribution
- Company distribution
- State distribution
- narrative length
- missingness
- vocabulary shift

### 2. Target Drift

Is the operational process changing?

Monitor:

- median delay
- P90
- P95
- >7d rate
- >14d rate
- >30d rate

### 3. Model Drift

Is prediction behavior changing?

Monitor:

- prediction distribution
- prediction mean
- prediction median
- prediction P95
- residual distribution

### 4. Business Drift

Is the model still helping operations?

Monitor:

- SLA breach rate
- alert workload
- useful alert rate
- analyst override rate
- operational outcomes

In [62]:
## Retraining Trigger Framework
retraining_triggers = {
    "target_distribution_shift": True,
    "mae_degradation": True,
    "tail_recall_degradation": True,
    "prediction_distribution_shift": True,
    "new_company_distribution": True,
    "business_kpi_degradation": True
}

pd.Series(retraining_triggers)

target_distribution_shift        True
mae_degradation                  True
tail_recall_degradation          True
prediction_distribution_shift    True
new_company_distribution         True
business_kpi_degradation         True
dtype: bool

## Model Lifecycle

Research
   ↓
Offline Validation
   ↓
Frozen Test
   ↓
Shadow Deployment
   ↓
Human Review
   ↓
Controlled Pilot
   ↓
Production
   ↓
Continuous Monitoring
   ↓
Retraining / Retirement

In [63]:
# — Final Project 2 Decision

final_business_decision = {
    "model": "One-stage Ridge log1p",
    
    "frozen_test_status": "FAILED_TO_BEAT_BASELINE",
    
    "production_status": "NOT_APPROVED",
    
    "recommended_use": [
        "research benchmark",
        "error analysis",
        "future model comparison",
        "shadow evaluation candidate"
    ],
    
    "baseline_status": (
        "retain as operational benchmark"
    ),
    
    "next_model_direction": [
        "better tail modeling",
        "regime-aware architecture",
        "process-state features if available at intake",
        "probabilistic prediction",
        "repeated temporal validation",
        "cost-sensitive SLA risk modeling"
    ]
}

pd.Series(final_business_decision)

model                                               One-stage Ridge log1p
frozen_test_status                                FAILED_TO_BEAT_BASELINE
production_status                                            NOT_APPROVED
recommended_use         [research benchmark, error analysis, future mo...
baseline_status                           retain as operational benchmark
next_model_direction    [better tail modeling, regime-aware architectu...
dtype: object

## Final Analysis Insights

#### 1. The model did not win

The final Ridge model did not beat the simple baseline on frozen test
data.

This is an important result, not a failure of the project.

---

#### 2. The process is highly non-stationary

The test period contained dramatically fewer long-delay cases than
the training period.

Therefore a model trained on historical delay behavior faced a
different operational regime.

---

#### 3. Prediction is not enough

A production Data Science system needs:

Prediction
→ Decision
→ Action
→ Outcome
→ Feedback

---

#### 4. Baselines are business assets

The baseline should remain permanently available.

A complex model should earn its place by demonstrating incremental
value over a simple policy.

---

#### 5. Current recommendation

Do NOT deploy the Ridge model for autonomous triage prioritization.

Retain it as an experimental benchmark and use it to guide the next
model iteration.

---

#### 6. Future modeling direction

The next serious modeling iteration should focus on the underlying
structure discovered during EDA:

mostly-immediate processing + rare long-delay regime.

Potential future architecture:

Stage 1:
Probability of SLA-risk / delayed regime

Stage 2:
Conditional delay distribution

Stage 3:
Operational decision policy

rather than treating the entire process as one stationary regression
problem.